In [12]:
import os ,io , time , json , re ,ast
import pandas as pd 
import requests
from google import genai 
from google.genai import types
from tenacity import retry , stop_after_attempt , wait_exponential 
from openpyxl import load_workbook 
from pdf2image import convert_from_path , pdfinfo_from_path 
from PIL import Image 
from dotenv import load_dotenv
load_dotenv()
client = genai.Client(api_key = os.getenv("Gemini_Api_Key2"))
MODEL = 'gemini-3.5-flash-lite'

Constituency_Folder = r'C:\Users\K Naveen\OneDrive\Desktop\SAIVED\Malakpet_constituency'
POPPLER_PATH = r'C:\Users\K Naveen\OneDrive\Desktop\SAIVED\poppler-26.02.0\Library\bin'

Processed_Booths = 'Processed_Booths.txt'
Raw_Output = 'Raw_Output.xlsx'
Review_Output = 'Review_Output.xlsx'
Booths_address = 'Booth_address.xlsx'

print(f'Environment Setup Completed')

Environment Setup Completed


In [3]:
RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties":{
        "page_type":{"type":"STRING","enum":["COVER","DATA","OTHER"]},
        "part_no":{"type":"STRING"},
        "polling_station_no_and_name":{"type":"STRING"},
        "polling_station_address":{"type":"STRING"},
        "section_no_and_name":{"type":"STRING"},
        "voters":{
            "type":"ARRAY",
            "items":{
                "type":"OBJECT",
                "properties":{
                    "sr_no":{"type":"STRING"},
                    "voter_id":{"type":"STRING"},
                    "voter_name":{"type":"STRING"},
                    "age":{"type":"STRING"},
                    "gender":{"type":"STRING"},
                    "relative_name":{"type":"STRING"},
                    "relation":{"type":"STRING"},
                    "house_number":{"type":"STRING"},
                    "religion_inference_by_name":{"type":"STRING","enum":["Hindu","Sikh","Christian","Muslim","Unknown"]},
                    "status":{"type":"STRING","enum":["Active","Deleted"],}
                    
                
            },
            "required":["sr_no","voter_id","voter_name","age","gender","relative_name",
                            "house_number","relation","religion_inference_by_name","status"]
                
        }
    }
    
    },
    "required":["page_type"]
}



In [4]:
PROMPT_TEXT =  """Act as an Electoral Roll Data Extraction Expert.
First, classify this page:
- Set page_type to 'COVER' if this is the first/summary page showing
  'Polling station details' with a numbered polling station and its address.
- Set page_type to 'DATA' if this page contains a grid of voter boxes with
  details like Name, Father/Husband Name, House No, Age, Gender, and Voter ID.
- Set page_type to 'OTHER' if this is a map view, photo summary page,
  elector summary table, or signature page.

If page_type is 'COVER':
1. From the section '3. Polling station details', extract:
   - polling_station_no_and_name as the EXACT combined text under
     'No. and Name of Polling Station' (e.g., '3 - Saidabad').
   - polling_station_address as the EXACT literal text under
     'Address of Polling Station'. Do not paraphrase, complete, or shorten it.
2. Also read Part No from the top header (e.g., 'Part No. : 3').
3. Leave the voters array empty.

If page_type is 'OTHER', leave the voters array empty.

If page_type is 'DATA':
1. Read Part No from the top-right header (e.g., 'Part No. : 3').
2. Read section_no_and_name from the top-left sub-header, keeping the
   number and name together exactly as printed
   (e.g., 'Section No and Name 1-Zakir Hussan Colony' -> '1-Zakir Hussan Colony').
3. Extract ALL voter boxes row-by-row, left-to-right.
4. If a box has a 'DELETED' or 'S2' watermark stamped across it,
   set status = 'Deleted', otherwise 'Active'.
5. Extract house_number as exact literal text (e.g., '16-1-14/5/1').
6. Infer religion strictly from the voter's primary name:
   - Hindu, Muslim, Christian, Sikh, or Unknown (if ambiguous/Jain/Buddhist -> Hindu).
"""

In [5]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def voter_data_extraction (pil_image):
    buffer = io.BytesIO()
    pil_image.save(buffer,format='JPEG',quality=85)
    Image_bytes=buffer.getvalue()

    response=client.models.generate_content(
        model=MODEL,

        contents=[
        types.Part.from_bytes(data=Image_bytes, mime_type='image/jpeg'),
        PROMPT_TEXT
        ],
        config=types.GenerateContentConfig
        (response_mime_type = "application/json",
         response_schema = RESPONSE_SCHEMA,
         temperature = 0.1)
            
    )
    return json.loads(response.text)

In [6]:
def validate_row(row):
    flags = []

    sr_no = str(row.get("sr_no", '')).strip()
    voter_id = str(row.get("voter_id", '')).strip()
    voter_name = str(row.get("voter_name", '')).strip()

    
    age = str(row.get("age", '')).strip()
    if age.endswith('.0'):
        age = age[:-2]

    gender = str(row.get("gender", '')).strip()
    relative_name = str(row.get("relative_name", '')).strip()
    relation = str(row.get("relation", '')).strip()
    house_number = str(row.get("house_number", '')).strip()
    status = str(row.get("status", '')).strip()
    religion_inference_by_name = str(row.get("religion_inference_by_name", '')).strip()

    if not sr_no.isdigit():
        flags.append("invalid_sr_no")

    if not re.match(r'^[A-Z]{3}\d{7}$', voter_id):
        flags.append("invalid_voter_id")

    if len(voter_name) < 2:
        flags.append("missing_voter_name")

    if voter_name.lower() == relative_name.lower() and len(voter_name) > 1:
        flags.append("voter_name_equals_relative_name")

    if not age.isdigit() or not (18 <= int(age) <= 120):
        flags.append("invalid_age")

    if gender not in ("Male", "Female", "Third Gender"):
        flags.append("invalid_gender")

    if len(relative_name) < 2:
        flags.append("missing_relative_name")

    
    if relation not in ("Father", "Husband", "Mother", "Others",
                        "Fathers", "Husbands", "Mothers",
                        "Fathers Name", "Husbands Name", "Mothers Name", "Others Name",
                        "Other"):
        flags.append("invalid_relation")

    if len(house_number) < 2:
        flags.append("invalid_house_number")

    if status not in ("Active", "Deleted"):
        flags.append("invalid_status")

    if religion_inference_by_name not in ("Hindu", "Muslim", "Sikh", "Christian", "Unknown"):
        flags.append("invalid_religion")

    return flags

In [7]:
def save_excel_as_safe (df , filename,text_columns=["house_number"]):
    if df.empty:
        return

    df.to_excel(filename,index=False)
    wb = load_workbook(filename)
    ws = wb.active

    for column_name in text_columns:
        if column_name in df.columns:
            column_index = list(df.columns).index(column_name)+1
            for row in range(2 , ws.max_row + 1):
                ws.cell(row=row , column=column_index).number_format = "@"
    wb.save(filename)
        

In [8]:
already_done = set() 
if os.path.exists(Processed_Booths):
    with open (Processed_Booths , 'r') as f:
        already_done = set(f.read().splitlines())

if os.path.exists(Raw_Output):
    master_df = pd.read_excel(Raw_Output)
    master_rows = master_df.to_dict('records')
else:
    master_rows = []

if os.path.exists(Booths_address):
    address_df = pd.read_excel(Booths_address)
    address_rows = address_df.to_dict('records')
else:
    address_rows = [] 

booth_file = [f for f in os.listdir(Constituency_Folder) if f.lower().endswith('.pdf')]

for booht_index , file_name in enumerate( booth_file , start = 1):
    if file_name in already_done:
        continue

    pdf_path = os.path.join(Constituency_Folder , file_name)

    try:
        info = pdfinfo_from_path(pdf_path , poppler_path = POPPLER_PATH)
        total_pages = info['Pages']
        booth_row = [] 
        booth_meta = {}
        failed_pages = [] 

        for page_num in range(1 , total_pages + 1):
            page = convert_from_path(pdf_path , first_page = page_num , last_page = page_num,
                                     dpi = 150 , poppler_path = POPPLER_PATH)

            image_page = page[0]

            try: 
                data =  voter_data_extraction (image_page)
                if data.get("page_type")=="COVER":
                    booth_meta={
                        'part_no':data.get('part_no','NA'),
                        'polling_station_no_and_name':data.get('polling_station_no_and_name','NA'),
                        'polling_station_address':data.get('polling_station_address'),
                        'booth_file':file_name}
                if data.get("page_type")=="DATA" and "voters" in data:
                    part_no = data.get('part_no','NA')
                    section_no_and_name = data.get('section_no_and_name','NA')
                    for v in data["voters"]:
                        v['part_no']=part_no
                        v['section_no_and_name']= section_no_and_name
                        v['polling_station_no_and_name']=booth_meta.get('polling_station_no_and_name','NA')
                        v['polling_station_address']=booth_meta.get('polling_station_address','NA')
                        v['flags']=validate_row(v)
                        v['booth_file']=file_name
                        v['page_no']=page_num
                        booth_row.append(v)
                
                time.sleep(3)

            except Exception as e :
                print(f'Error on page {page_num}:{e}')
                failed_pages.append(page_num)

        if booth_row and not failed_pages:
            master_rows.extend(booth_row)
            with open(Processed_Booths, 'a') as f:
                  f.write(file_name + '\n')  
            master_df = pd.DataFrame(master_rows)
            save_excel_as_safe(master_df , Raw_Output)
        elif failed_pages:
            print (f'{file_name}: {len(failed_pages)} pages failed - Marked as not done will try it in the next run')
    

        if booth_meta and not failed_pages:
            address_rows.append(booth_meta)
            address_df = pd.DataFrame(address_rows)
            save_excel_as_safe(address_df ,Booths_address ,text_columns=["polling_station_address"] )

    except Exception as e:
        print(f'Failed_booth :{file_name}:{e}')
        


In [11]:
df = pd.read_excel(Raw_Output)
df['age'] = df['age'].apply(lambda x: str(int(x)) if pd.notna(x) and str(x).replace('.','').replace('-','').isdigit() else str(x))
df['flags'] = df.apply(lambda row: validate_row(row.to_dict()), axis=1)
needs_review = df[df['flags'].apply(len) > 0]

save_excel_as_safe(clean, Clean_Output, text_columns=["house_number", "age"])
save_excel_as_safe(needs_review, Review_Output, text_columns=["house_number", "age"])

print(f'Total: {len(df)} | Review: {len(needs_review)}')
print(df[df['flags'].apply(len) > 0]['flags'].astype(str).value_counts().head(10))


KeyboardInterrupt

